# C.4 — Tính loss DPO, gradient và ảnh hưởng của β
**Đề tài 9, tr. 19, C.4.** Đề yêu cầu tính từ bốn log-xác suất nhưng không ghi bộ số cụ thể. Các số dưới đây do nhóm dùng **để minh họa phép tính**, không phải số giảng viên cấp, đầu ra của một checkpoint hay nhãn người chấm ở D.4. Có thể thay bốn log-xác suất và β trong ô tính bên dưới nếu nhận đầu vào khác. Chạy tuần tự trên CPU bằng môi trường PyTorch A.1; không cần dữ liệu, mạng hoặc GPU.

## 1. Lời giải cho một cặp bất kỳ
Cùng một prompt $x$, gọi $w$ là đáp án được ưa thích, $l$ là đáp án bị từ chối. Đặt $p_i=\ln\pi_\theta(y_i\mid x)$, $r_i=\ln\pi_{\rm ref}(y_i\mid x)$ với $i\in\{w,l\}$. Đây là **tổng** log-xác suất tự nhiên của token đáp án (không gồm prompt hay padding), không phải trung bình theo chiều dài như trong perplexity. Các log-xác suất hữu hạn; $\beta>0$, reference đóng băng.

Theo [B.5](../docs/theory/part-b.pdf) và [Rafailov et al., §4, Eq. (7)](../references/DPO.pdf), xác suất Bradley–Terry dự đoán nhãn $w\succ l$ là $\sigma(z)$, với
$$m=(p_w-p_l)-(r_w-r_l),\qquad z=\beta m,\qquad L=-\ln\sigma(z)=\ln(1+e^{-z}).$$
$m$ so **tỷ số xác suất** của hai đáp án giữa policy và reference, không chỉ so $p_w$ với $p_l$. Xác suất ưu tiên $\sigma(z)$ khác xác suất policy sinh riêng đáp án $w$.

Vì $\mathrm dL/\mathrm dz=-\sigma(-z)$ và $\partial z/\partial p_w=\beta=-\partial z/\partial p_l$, đặt $g=\beta\sigma(-z)$ thì
$$\frac{\partial L}{\partial p_w}=-g,\qquad \frac{\partial L}{\partial p_l}=+g,\qquad \nabla_\theta L=-g\big(\nabla_\theta p_w-\nabla_\theta p_l\big).$$
Hạ gradient khuyến khích tăng **chênh lệch** log-xác suất của policy theo hướng nhãn ưu tiên. Hai hệ số $-g,+g$ tính được từ bốn số đầu vào; để tính vector gradient trên từng trọng số $\theta$ còn cần đạo hàm của mô hình. Nếu xét hai log-xác suất reference như biến toán học độc lập thì đạo hàm lần lượt là $+g,-g$, nhưng khi huấn luyện không truyền gradient vào reference. Với loss trung bình trên $B$ cặp, mỗi cặp đóng góp thêm hệ số $1/B$.

## 2. Thế số bằng tay — ví dụ minh họa
Chọn bốn log-xác suất $(p_w,p_l,r_w,r_l)=(\ln 0{,}2,\ln 0{,}1,\ln 0{,}4,\ln 0{,}1)$ và $\beta=1$. Hai đáp án không phải toàn bộ không gian đầu ra, nên hai xác suất mỗi mô hình không cần cộng thành 1.

Policy ưu tiên $w$ so với $l$ theo tỷ số $2:1$, nhưng reference ưu tiên mạnh hơn, $4:1$. Vậy $m=\ln 2-\ln 4=-\ln 2$, $z=-\ln 2$. Suy ra $\sigma(z)=1/3$, **$L=\ln 3\approx1{,}098612$**, và $g=2/3$. Gradient theo $(p_w,p_l)$ là **$(-2/3,+2/3)$**. Loss lớn hơn $\ln 2$ dù $p_w>p_l$: policy chưa ưu tiên $w$ mạnh bằng reference.

In [ ]:
from pathlib import Path
import math
import platform
import sys
import torch

here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / 'src/dpo.py').is_file()), None)
if root is None:
    raise RuntimeError('Mở notebook từ gốc repo hoặc thư mục notebooks/.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.dpo import dpo_loss

print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | CPU, float64')

def check_pair(logps, beta):
    pw, pl, rw, rl = logps
    margin = (pw - pl) - (rw - rl)
    z = beta * margin
    # Hai dạng ổn định số cho softplus(-z) và sigmoid(-z).
    e = math.exp(-abs(z))
    loss = max(-z, 0.) + math.log1p(e)
    g = beta * ((e if z >= 0 else 1.) / (1. + e))
    inputs = tuple(torch.tensor(x, dtype=torch.float64, device='cpu', requires_grad=True) for x in logps)
    actual = dpo_loss(*inputs, beta=beta)
    gw, gl, grw, grl = torch.autograd.grad(actual, inputs, allow_unused=True)
    assert grw is None and grl is None, 'Reference nhận gradient ngoài ý muốn'
    for observed, expected in zip((actual.item(), gw.item(), gl.item()), (loss, -g, g)):
        assert math.isclose(observed, expected, rel_tol=1e-12, abs_tol=1e-14), (observed, expected)
    return margin, loss, -g, g

In [ ]:
# Ô thay đầu vào: nhập trực tiếp log-xác suất, không lấy log lần hai.
# Giá trị mặc định là ví dụ minh họa ở mục 2.
logps = (math.log(0.2), math.log(0.1), math.log(0.4), math.log(0.1))
beta = 1.0
m, loss, gw, gl = check_pair(logps, beta)
print(f'm = {m:.8f}; z = {beta*m:.8f}; L = {loss:.8f}; dL/dp_w = {gw:+.8f}; dL/dp_l = {gl:+.8f}')

## 3. Thay β nhưng giữ nguyên bốn log-xác suất
Ở **một cặp cố định**, $\partial L/\partial\beta=-m\sigma(-\beta m)$. Khi $\beta\to0^+$, loss tiến tới $\ln 2$, còn $g\sim\beta/2$: tín hiệu gradient yếu cho mọi cặp.

| Policy so với reference theo nhãn ưu tiên | Khi tăng β | Khi β rất lớn |
|---|---|---|
| $m<0$: chênh lệch chưa đủ | Loss tăng; độ lớn gradient $g$ tăng | $L\sim\beta\lvert m\rvert$, $g\sim\beta$ |
| $m=0$: hai tỷ số bằng nhau | $L=\ln 2$ không đổi, nhưng gradient **không bằng 0** | $g=\beta/2$ |
| $m>0$: chênh lệch đã vượt reference | Loss giảm; $g$ không tăng đơn điệu | $L\to0$, $g\to0$ do sigmoid bão hòa |

Kiểm bằng ba ví dụ: giữ reference $(0{,}4;0{,}1)$ và xác suất policy cho $l$ bằng $0{,}1$. Chỉ đổi xác suất policy cho $w$ thành $0{,}2$, $0{,}4$, $0{,}8$ để thu được $m=-\ln 2,0,+\ln 2$. Mỗi hàng kết quả là một cặp riêng, không lấy trung bình giữa ba ví dụ.

In [ ]:
print('      m    beta        loss      dL/dp_w      dL/dp_l')
for pw_prob in (0.2, 0.4, 0.8):
    example = tuple(map(math.log, (pw_prob, 0.1, 0.4, 0.1)))
    for b in (0.1, 1., 2., 8.):
        m_i, l_i, gw_i, gl_i = check_pair(example, b)
        print(f'{m_i:7.4f} {b:7.1f} {l_i:11.7f} {gw_i:12.7f} {gl_i:12.7f}')

**Phân biệt hai cách đọc β.** Bảng trên chỉ đổi β tại policy cố định. Trong mục tiêu ban đầu $\mathbb E[R]-\beta\,\mathrm{KL}(\pi\Vert\pi_{\rm ref})$, khi giữ reward cố định thì β lớn phạt lệch reference mạnh hơn; nghiệm tối ưu lúc đó cũng thay đổi. Không dùng bảng minh họa để chọn β cho D.4 hoặc kết luận DPO cải thiện chất lượng. Nếu giảng viên cấp bốn log-xác suất riêng, ghi nguồn số liệu và thay ô đầu vào mục 2; phần đối chiếu A.4 và công thức giữ nguyên.